# Physicochemical properties

This notebook demonstrates some of the higher-level analysis tools from MDTraj that focus on the physicochemical properties of a macromolecule.
These can be useful to understand (de)stabilizing factors.
A complete overview of analysis functions in MDTraj can be found here: https://www.mdtraj.org/development/analysis.html

The analysis uses the output files `init.pdb`, `scalars.csv` and `traj.dcd` of the villin headpiece simulation in [../04_protein/01_villin_headpiece.ipynb](../04_protein/01_villin_headpiece.ipynb).
These files must first be copied to this directory, as explained at the end of that notebook.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import mdtraj
import numpy as np
import pandas

The next cell loads the trajectory and strips water molecules, because all analyses below only concern the protein.
This also makes the analysis faster.

In [ ]:
traj = mdtraj.load("traj.dcd", top="init.pdb")
traj.restrict_atoms(traj.topology.select("protein"))
df = pandas.read_csv("scalars.csv")

## 1. Hydrogen bonding

(Internal) hydrogen bonds are one of the important factors that stabilize protein folds and also in other macromolecular systems they often play an important role.
We will use the [Wernet-Nilsson criterion](https://www.mdtraj.org/development/api/generated/mdtraj.wernet_nilsson.html) implemented in MDTraj to count the number of hydrogen bonds in each frame.
Because the water molecules were removed, only hydrogen bonds within the protein are counted.

In [ ]:
hbonds = mdtraj.wernet_nilsson(traj)
numhbonds = [len(pairs) for pairs in hbonds]
plt.close("numh")  # This is needed to rerun the code cell correctly
fig, ax1 = plt.subplots(num="numh")
ax1.plot(df["Time (ps)"], numhbonds, color="C0")
ax1.set_xlabel("Time [ps]")
ax1.set_ylabel("Number of hydrogen bonds", color="C0")
ax1.tick_params(axis="y", labelcolor="C0")
ax2 = ax1.twinx()
ax2.plot(df["Time (ps)"], df["Potential Energy (kJ/mole)"], color="C1")
ax2.set_ylabel("Potential energy [kJ/mol]", color="C1")
ax2.tick_params(axis="y", labelcolor="C1")

The potential energy reaches its equilibrium value faster than the number of hydrogen bonds.
This shows (again) that one should not just trust a single property to determine the equilibration phase.
Most of the increase of the potential energy is simply the thermal activation of the solvent, which is not necessarily representative for the solute.

## 2. Secondary structure assignment

The secondary structure assignment of every residue (at each time step) can be obtained with the [compute_dssp](https://www.mdtraj.org/development/api/generated/mdtraj.compute_dssp.html) function of MDTraj.
For small proteins like the villin headpiece, the results can be plotted conveniently with Matplotlib.

In [ ]:
dssp = mdtraj.compute_dssp(traj)
# Convert string characters to numbers
dssp_num = np.vectorize("HEC".index)(dssp)
plt.close("dssp")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="dssp")
im = ax.pcolormesh(df["Time (ps)"], np.arange(dssp.shape[1]), dssp_num.T, shading="nearest")
cbar = fig.colorbar(im, values=[0, 1, 2], ticks=[0, 1, 2])
cbar.ax.set_yticklabels(["H", "E", "C"])
ax.set_xlabel("Time [ps]")
ax.set_ylabel("Residue index")

This example already shows nicely the presence of the three alpha helices and how they change somewhat over time.

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Modify the above example to plot the 8-level DSSP assignment, instead of the simplified 3-level assignment.

## 3. Radial distribution function

The coordination of a functional group by another chemical moiety can be characterized with a [radial distribution function](https://en.wikipedia.org/wiki/Radial_distribution_function) (RDF).
In general, a radial distribution function is a ratio of two probability densities as function of an inter-atomic distance:

- In the numerator, one puts the empirically observed probability density for finding a given pair of atoms.
- The denominator contains the probability in case the two atoms would be uniformly distributed over space.

In case of dense liquids, the RDF converges to one for large distances because there is no long-range structure in a normal liquid.
In case of a single isolated system (like a protein), the RDF converges to zero because all empirical inter-atomic distances are bound by the size of the molecule.

We can use [compute_rdf](https://www.mdtraj.org/development/api/generated/mdtraj.compute_rdf.html) in combination with [select_pairs](https://www.mdtraj.org/development/api/generated/mdtraj.Topology.html) to easily generate RDFs.

In the example below, we compute the RDF for all backbone amide nitrogens (`N`) with backbone carbonyl oxygens (`O`).

In [ ]:
r, gr = mdtraj.compute_rdf(
    traj,
    traj.topology.select_pairs("protein and name N", "protein and name O"),
    r_range=(0.1, 0.5),
    bin_width=0.0005,
)
plt.close("rdf")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="rdf")
ax.plot(r, gr)
ax.set_xlabel("Radius [nm]")
ax.set_ylabel("Radial distribution function $g(r)$")

The sharp peak at approx $0.225\,\mathrm{nm}$ is not due to hydrogen bonds.
It comes from the `N` and `O` atoms in the same peptide bond (`O=C-N`), whose distance is fixed by the covalent geometry.
The broad shoulder around $0.29\,\mathrm{nm}$ is due to the backbone hydrogen bonds `N-H...O=C`, mostly between residues $i$ and $i+4$ in the alpha helices.
The peak around $0.35\,\mathrm{nm}$ mainly involves `N` and `O` atoms of the same residue or of nearby residues, whose distance depends on the backbone dihedral angles.
This example shows that one should be careful when interpreting an RDF of atoms within the same molecule: it also contains contributions from atom pairs whose distance is determined by covalent bonds.

## 4. Solvent accessible surface area

The Shrake-Rupley method can be used to measure the solvent-accessible surface area of each residue, showing which are exposed to the solvent water molecules.
For this, we use the [shrake_rupley](https://www.mdtraj.org/development/api/generated/mdtraj.shrake_rupley.html) function from MDTraj.
Because this calculation is relatively slow, it is only applied to every tenth frame.

In [ ]:
sasa = mdtraj.shrake_rupley(traj[::10], mode="residue")
plt.close("area")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="area")
im = ax.pcolormesh(df["Time (ps)"][::10], np.arange(traj.n_residues), sasa.T, shading="nearest")
cbar = fig.colorbar(im)
cbar.set_label("Solvent accessible surface area [nm$^2$]")
ax.set_xlabel("Time [ps]")
ax.set_ylabel("Residue index")

The three phenylalanine residues (PHE6, PHE10 and PHE17, i.e. residue indices 5, 9 and 16 in the plot) have the smallest solvent accessible surface area.
They form the hydrophobic core of the villin headpiece, which holds the three alpha helices together.
Most of the charged residues (lysine, arginine and glutamate), on the other hand, are strongly exposed to the solvent.